In [1]:
import pandas as pd
import scipy
import data_profiling
from data_profiling import ProfileReport
import numpy as np

In [2]:
uk_prices_2026 = pd.read_csv("UK_elec_data_1y_Oct2026.csv", sep=";")
uk_prices_hist = pd.read_csv("United_Kingdom.csv")
exchange_rates = pd.read_csv("EURO_GBP_rates.csv")
day_ahead_demand = pd.read_csv("dayahead_demand.csv") # check if always before 09:20
day_ahead_wind = pd.read_csv("dayahead_wind.csv") # check if always before 09:20

In [3]:
uk_prices_2026.head() # CET (so GMT +01:00)

,Id,Day,Hour,Period,Indicator,Zone name,Zone code,Key data,Value,Unit
0,147763592,2025-10-29,00:00:00,hourly,price,United Kingdom,UK,NaN,56.239570917759,€/MWh
1,147763593,2025-10-29,01:00:00,hourly,price,United Kingdom,UK,NaN,57.967818831943,€/MWh
2,147763594,2025-10-29,02:00:00,hourly,price,United Kingdom,UK,NaN,45.095351609058,€/MWh
3,147763595,2025-10-29,03:00:00,hourly,price,United Kingdom,UK,NaN,28.599523241955,€/MWh
4,147763596,2025-10-29,04:00:00,hourly,price,United Kingdom,UK,NaN,36.328963051251,€/MWh


In [4]:
ukp = uk_prices_2026[["Day", "Hour", "Value"]]
ProfileReport(ukp, minimal=True)
;

''

In [5]:
ukp = ukp.replace("€/MWh", np.nan)
ukp["Hour"] = pd.to_datetime(
    ukp["Day"].astype(str) + " " + ukp["Hour"].astype(str)
    )
ukp_hours = ukp[["Hour", "Value"]]
ukp_hours["Hour"] = (
    ukp_hours["Hour"]
    .dt.tz_localize("Etc/GMT-1")
    .dt.tz_convert("Europe/London")
)

C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\854513504.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ukp_hours["Hour"] = (


In [6]:
day_ahead_demand.head() # UK local time

,DAYSAHEAD,TARGETDATE,FORECASTDEMAND,CARDINALPOINT,CP_TYPE,CP_ST_TIME,CP_END_TIME,F_Point,FORECAST_TIMESTAMP
0,1,2018-01-02,24920,1F,F,30,30,NaN,2018-01-01T08:46:43Z
1,1,2018-01-02,25250,1A,P,100,230,NaN,2018-01-01T08:46:43Z
2,1,2018-01-02,23390,03,F,300,300,NaN,2018-01-01T08:46:43Z
3,1,2018-01-02,22480,1B,T,430,730,Om,2018-01-01T08:46:43Z
4,1,2018-01-02,37190,2F,P,800,900,NaN,2018-01-01T08:46:43Z


In [7]:
# cleaning demand - removing extraneous columns, changing 2400 to 0000 next day

demand = day_ahead_demand[["TARGETDATE", "FORECASTDEMAND", "CP_ST_TIME", "CP_END_TIME", "FORECAST_TIMESTAMP"]]

time_str = demand["CP_ST_TIME"].astype(str).str.zfill(4)
time_str_end = demand["CP_END_TIME"].astype(str).str.zfill(4)

midnight = time_str == "2400"
time_str = time_str.mask(midnight, "0000")
midnight_end = time_str_end == "2400"
time_str_end = time_str_end.mask(midnight_end, "0000")


demand["CP_ST_TIME"] = pd.to_datetime(
    demand["TARGETDATE"].astype(str) + " " +
    time_str.str[:2] + ":" + time_str.str[2:] + ":00")

demand["CP_END_TIME"] = pd.to_datetime(
    demand["TARGETDATE"].astype(str) + " " +
    time_str_end.str[:2] + ":" + time_str_end.str[2:] + ":00")

demand.loc[midnight, "CP_ST_TIME"] += pd.Timedelta(days=1)
demand.loc[midnight_end, "CP_END_TIME"] += pd.Timedelta(days=1)

demand = demand.drop(columns = ["TARGETDATE"])

C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\2757015363.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand["CP_ST_TIME"] = pd.to_datetime(
C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\2757015363.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand["CP_END_TIME"] = pd.to_datetime(
C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\2757015363.py:22: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit c

In [8]:
hours = pd.date_range(
    demand["CP_ST_TIME"].min().floor("h"),
    demand["CP_END_TIME"].max().ceil("h"),
    freq="h"
)

start = demand[["CP_ST_TIME", "FORECASTDEMAND"]].rename(
    columns={"CP_ST_TIME": "Endpoint"}
)
end = demand[["CP_END_TIME", "FORECASTDEMAND"]].rename(
    columns={"CP_END_TIME": "Endpoint"}
)

endpoints = pd.concat([start, end], ignore_index=True)
endpoints = endpoints.sort_values("Endpoint")

# Find the closest endpoint for each hour
hourly_df = pd.DataFrame({"Hour": hours})
hourly_df["Endpoint"] = hourly_df["Hour"]

hourly_df = pd.merge_asof(
    hourly_df.sort_values("Hour"),
    endpoints.sort_values("Endpoint"),
    left_on="Hour",
    right_on="Endpoint",
    direction="nearest",
    allow_exact_matches=True
)

In [9]:
demand_hours = hourly_df[["Hour", "FORECASTDEMAND"]]
demand_hours["Hour"] = (
    demand_hours["Hour"]
    .dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")
)
demand_hours.head()

C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\1745701569.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand_hours["Hour"] = (


,Hour,FORECASTDEMAND
0,2018-01-02 00:00:00+00:00,24920
1,2018-01-02 01:00:00+00:00,25250
2,2018-01-02 02:00:00+00:00,25250
3,2018-01-02 03:00:00+00:00,23390
4,2018-01-02 04:00:00+00:00,22480


In [10]:
day_ahead_wind.head() # GMT

,Datetime_GMT,Date,Settlement_period,Capacity,Incentive_forecast,Forecast_Timestamp
0,2018-04-16T23:00:00,2018-04-17,1,11173,7523,2018-04-19T08:05:35
1,2018-04-16T23:30:00,2018-04-17,2,11173,7656,2018-04-19T08:05:35
2,2018-04-17T00:00:00,2018-04-17,3,11173,7736,2018-04-19T08:05:35
3,2018-04-17T00:30:00,2018-04-17,4,11173,7765,2018-04-19T08:05:35
4,2018-04-17T01:00:00,2018-04-17,5,11173,7852,2018-04-19T08:05:35


In [11]:
wind = day_ahead_wind[["Datetime_GMT", "Capacity", "Incentive_forecast"]]
ProfileReport(wind)
;

''

In [12]:
wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])
wind_hours = wind[wind["Datetime_GMT"].dt.minute == 0]
wind_hours["Hour"] = (
    wind_hours["Datetime_GMT"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/London")
)
wind_hours = wind_hours.drop(columns = ["Datetime_GMT"])

C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\3813093146.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])
C:\Users\hamis\AppData\Local\Temp\ipykernel_31868\3813093146.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wind_hours["Hour"] = (


In [13]:
df = (
    ukp_hours
    .merge(demand_hours, left_on="Hour", right_on="Hour", how="left")
    .merge(wind_hours, left_on="Hour", right_on="Hour", how="left")
)

In [14]:
df.head()

,Hour,Value,FORECASTDEMAND,Capacity,Incentive_forecast
0,2025-10-28 23:00:00+00:00,56.239570917759,22699.0,23534.0,13491.0
1,2025-10-29 00:00:00+00:00,57.967818831943,22699.0,23548.0,13351.0
2,2025-10-29 01:00:00+00:00,45.095351609058,22790.0,23548.0,13367.0
3,2025-10-29 02:00:00+00:00,28.599523241955,22790.0,23548.0,13455.0
4,2025-10-29 03:00:00+00:00,36.328963051251,22790.0,23548.0,13357.0


In [ ]:
ProfileReport(df)
;

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 50.49it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]